In [1]:
import streamlit as st

# 1. Add a title to your web page
st.title("SG LNG ML Prototype 🚀")

# 2. Add some simple text
st.write("Welcome! Move the slider below to see how easy visualising data is.")

# 3. Add an interactive slider (Min value: 0, Max value: 100, Default: 25)
user_number = st.slider("Pick a number", 0, 100, 25)

# 4. Use the user's input to display a dynamic message
st.write(f"You picked: **{user_number}**!")
st.write(f"That number squared is: **{user_number ** 2}**")

2026-06-10 20:35:24.775 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:24.818 
  command:

    streamlit run /opt/anaconda3/lib/python3.13/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-06-10 20:35:24.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:24.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:24.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:24.819 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:24.819 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:24.819 Thread 'MainThread': m

In [2]:
# Load libraries

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import joblib
import os
from datetime import datetime

In [3]:
# PAGE CONFIGURATION

import streamlit as st

try:
    st.set_page_config(
        page_title="LNG Arbitrage Regime Classifier",
        page_icon="🚢",
        layout="wide",
        initial_sidebar_state="expanded",
    )
except st.errors.StreamlitAPIException:
    # set_page_config was already called (e.g. during hot-reload)
    # Safe to ignore.
    pass

2026-06-10 20:35:25.798 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [4]:
# CONSTANTS & LABEL MAP

LABEL_MAP = {
    0: "Europe Premium",
    1: "Neutral",
    2: "Asia Premium",
}

COLORS = {
    "Europe Premium": "#c0392b",   # deep red
    "Neutral":        "#e67e22",   # amber
    "Asia Premium":   "#2980b9",   # steel blue
}

FEATURES = [
    "ttf_price",
    "jkm_lag1",
    "jkm_lag3",
    "jkm_rolling_3m",
    "henry_price",
    "jkm_ttf_spread",
    "jkm_hh_spread",
    "lng_share",
]

FREIGHT        = 1.50   # USD/MMBtu typical Asia–Europe LNG shipping cost
SPLIT_DATE     = "2023-01-01"
MODEL_PATH     = "models/best_model.pkl"
SCALER_PATH    = "models/scaler.pkl"
DATA_PATH      = "data/lng_arbitrage_clean.csv"


In [5]:
# CUSTOM CSS

def inject_css():
    st.markdown("""
    <style>
    .block-container { padding-top: 1.5rem; }
    .pill-asia    { background:#2980b9; color:#fff; padding:4px 12px;
                    border-radius:999px; font-weight:600; font-size:0.9rem; }
    .pill-neutral { background:#e67e22; color:#fff; padding:4px 12px;
                    border-radius:999px; font-weight:600; font-size:0.9rem; }
    .pill-europe  { background:#c0392b; color:#fff; padding:4px 12px;
                    border-radius:999px; font-weight:600; font-size:0.9rem; }
    section[data-testid="stSidebar"] h3 {
        color: #2d2d2d; font-size: 0.85rem;
        text-transform: uppercase; letter-spacing: 0.05em;
    }
    </style>
    """, unsafe_allow_html=True)


In [6]:
# CACHED DATA & MODEL LOADERS

@st.cache_data
def load_data(path: str) -> pd.DataFrame:
    """Load the cleaned LNG arbitrage dataset."""
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index)
    return df


@st.cache_resource
def load_model(model_path: str, scaler_path: str):
    """Load the trained best model and its StandardScaler."""
    model  = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    return model, scaler

2026-06-10 20:35:25.805 No runtime found, using MemoryCacheStorageManager


In [7]:
# PREDICTION HELPER

def predict(model, scaler, feature_values: dict):
    """
    Parameters
    ----------
    feature_values : dict with keys matching FEATURES list,
                     values are raw (unscaled) floats.

    Returns
    -------
    label_int  : int
    label_str  : str
    proba_dict : dict
    """
    X_raw = pd.DataFrame([feature_values])[FEATURES]
    X_scaled = scaler.transform(X_raw)

    label_int = int(model.predict(X_scaled)[0])
    label_str = LABEL_MAP[label_int]

    # Probability output — only available if model has predict_proba
    if hasattr(model, "predict_proba"):
        probas = model.predict_proba(X_scaled)[0]
        proba_dict = {LABEL_MAP[i]: round(float(p), 4)
                      for i, p in enumerate(probas)}
    else:
        # Fallback: hard one-hot for models without predict_proba
        proba_dict = {name: (1.0 if name == label_str else 0.0)
                      for name in LABEL_MAP.values()}

    return label_int, label_str, proba_dict

In [8]:
# SIDEBAR: NAVIGATION & LIVE INPUT

with st.sidebar:
    st.image("https://img.icons8.com/fluency/64/lng-tanker.png", width=48)
    st.title("LNG Regime Classifier")
    st.caption("JKM–TTF Spread · Arbitrage Detection")
    st.divider()

    # ── Navigation ──────────────────────────────────────────
    st.subheader("Navigation")
    page = st.radio(
        label="page",
        options=["📊 Dashboard", "🔍 Live Predictor", "📈 Model Performance", "ℹ️ About"],
        label_visibility="collapsed",
    )
    st.divider()

    # ── Live Predictor Inputs ───────────────────────────────
    # These sliders are always visible so the user can tweak
    # values and immediately see the regime change on the
    # Live Predictor page.
    st.subheader("Feature Inputs")

    jkm_ttf_spread = st.slider(
        "JKM–TTF Spread ($/MMBtu)",
        min_value=-10.0, max_value=30.0, value=2.5, step=0.1,
        help="Primary discriminative feature. "
             "< -1.0 → Europe Premium | -1.0 to 1.5 → Neutral | > 1.5 → Asia Premium"
    )
    ttf_price = st.slider(
        "TTF Price ($/MMBtu)",
        min_value=0.0, max_value=80.0, value=12.0, step=0.5
    )
    jkm_lag1 = st.slider(
        "JKM Price 1-Month Lag ($/MMBtu)",
        min_value=0.0, max_value=80.0, value=14.0, step=0.5
    )
    jkm_lag3 = st.slider(
        "JKM Price 3-Month Lag ($/MMBtu)",
        min_value=0.0, max_value=80.0, value=13.5, step=0.5
    )
    jkm_rolling_3m = st.slider(
        "JKM 3-Month Rolling Avg ($/MMBtu)",
        min_value=0.0, max_value=80.0, value=13.8, step=0.5
    )
    henry_price = st.slider(
        "Henry Hub Price ($/MMBtu)",
        min_value=0.0, max_value=15.0, value=2.8, step=0.1
    )
    jkm_hh_spread = st.slider(
        "JKM–HH Spread ($/MMBtu)",
        min_value=-5.0, max_value=40.0, value=11.2, step=0.1
    )
    lng_share = st.slider(
        "LNG Share of Global Gas Trade (%)",
        min_value=0.0, max_value=60.0, value=34.0, step=0.5
    )

# Bundle sidebar inputs into a dict for the prediction helper
sidebar_inputs = {
    "ttf_price":      ttf_price,
    "jkm_lag1":       jkm_lag1,
    "jkm_lag3":       jkm_lag3,
    "jkm_rolling_3m": jkm_rolling_3m,
    "henry_price":    henry_price,
    "jkm_ttf_spread": jkm_ttf_spread,
    "jkm_hh_spread":  jkm_hh_spread,
    "lng_share":      lng_share,
}


2026-06-10 20:35:25.811 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.812 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.813 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.813 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.813 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.814 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.814 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.814 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [9]:
# LOAD DATA & MODEL 

data_ok  = os.path.exists(DATA_PATH)
model_ok = os.path.exists(MODEL_PATH) and os.path.exists(SCALER_PATH)

if data_ok:
    df = load_data(DATA_PATH)
else:
    st.warning(f"⚠️  Data file not found at `{DATA_PATH}`. "
               "Historical charts will be unavailable.", icon="⚠️")
    df = None

if model_ok:
    model, scaler = load_model(MODEL_PATH, SCALER_PATH)
else:
    st.warning(f"⚠️  Model files not found at `{MODEL_PATH}` / `{SCALER_PATH}`. "
               "Live prediction will be unavailable.", icon="⚠️")
    model, scaler = None, None



2026-06-10 20:35:25.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [10]:
# PAGE: DASHBOARD

if page == "📊 Dashboard":

    st.title("📊 LNG Arbitrage Regime Dashboard")
    st.caption("Monthly regime classification based on JKM–TTF spread dynamics · Jan 2014 – Apr 2026")

    if df is not None and "arbitrage_label" in df.columns:

        # ── KPI Metric Cards ─────────────────────────────────
        col1, col2, col3, col4 = st.columns(4)

        label_counts = df["arbitrage_label"].value_counts()
        latest_spread = df["jkm_ttf_spread"].iloc[-1]
        latest_date   = df.index[-1].strftime("%b %Y")

        col1.metric(
            label="🔵 Asia Premium Months",
            value=int(label_counts.get(2, 0)),
            help="Months where JKM–TTF spread > 1.50 $/MMBtu"
        )
        col2.metric(
            label="🟠 Neutral Months",
            value=int(label_counts.get(1, 0)),
            help="Months where -1.0 ≤ JKM–TTF spread ≤ 1.50 $/MMBtu"
        )
        col3.metric(
            label="🔴 Europe Premium Months",
            value=int(label_counts.get(0, 0)),
            help="Months where JKM–TTF spread < -1.0 $/MMBtu"
        )
        col4.metric(
            label=f"Latest Spread ({latest_date})",
            value=f"{latest_spread:.2f} $/MMBtu",
            delta=f"{(latest_spread - df['jkm_ttf_spread'].iloc[-2]):.2f} vs prior month"
        )

        st.divider()

        # ── Spread Over Time (coloured by regime) ────────────
        # Build one Scatter trace per regime so each gets its
        # own colour and legend entry.
        regime_names = {0: "Europe Premium", 1: "Neutral", 2: "Asia Premium"}
        fig_spread   = go.Figure()

        for label_id, regime_name in regime_names.items():
            mask = df["arbitrage_label"] == label_id
            fig_spread.add_trace(go.Scatter(
                x=df.index[mask],
                y=df.loc[mask, "jkm_ttf_spread"],
                mode="markers+lines",
                name=regime_name,
                marker=dict(color=COLORS[regime_name], size=6),
                line=dict(color=COLORS[regime_name], width=1),
            ))

        # Freight threshold lines
        fig_spread.add_hline(y=FREIGHT,  line_dash="dash", line_color="#444",
                             annotation_text=f"Asia threshold ({FREIGHT} $/MMBtu)",
                             annotation_position="top left")
        fig_spread.add_hline(y=-1.0, line_dash="dot", line_color="#888",
                             annotation_text="Europe threshold (−1.0 $/MMBtu)",
                             annotation_position="bottom left")

        fig_spread.update_layout(
            title="JKM–TTF Spread Over Time — Coloured by Arbitrage Regime",
            xaxis_title="Date",
            yaxis_title="Spread ($/MMBtu)",
            legend_title="Regime",
            hovermode="x unified",
            height=420,
        )
        st.plotly_chart(fig_spread, use_container_width=True)

        # ── Regime Distribution + Annual Heatmap ─────────────
        col_left, col_right = st.columns([1, 2])

        with col_left:
            # Horizontal bar chart — regime counts
            fig_dist = go.Figure(go.Bar(
                x=[int(label_counts.get(2, 0)),
                   int(label_counts.get(1, 0)),
                   int(label_counts.get(0, 0))],
                y=["Asia Premium", "Neutral", "Europe Premium"],
                orientation="h",
                marker_color=[COLORS["Asia Premium"],
                              COLORS["Neutral"],
                              COLORS["Europe Premium"]],
            ))
            fig_dist.update_layout(
                title="Regime Distribution (all months)",
                xaxis_title="Count",
                height=300,
                margin=dict(t=40, b=20),
            )
            st.plotly_chart(fig_dist, use_container_width=True)

        with col_right:
            # Annual regime heatmap: year × month → regime
            df_heat = df[["arbitrage_label"]].copy()
            df_heat["year"]  = df_heat.index.year
            df_heat["month"] = df_heat.index.month

            pivot = df_heat.pivot(index="year", columns="month",
                                  values="arbitrage_label")
            month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                            "Jul","Aug","Sep","Oct","Nov","Dec"]

            fig_heat = px.imshow(
                pivot,
                color_continuous_scale=[
                    [0.0, COLORS["Europe Premium"]],
                    [0.5, COLORS["Neutral"]],
                    [1.0, COLORS["Asia Premium"]],
                ],
                labels=dict(color="Regime"),
                x=month_labels[:pivot.shape[1]],
                y=pivot.index.tolist(),
                aspect="auto",
                title="Monthly Regime Heatmap (0=EP · 1=Neutral · 2=AP)",
            )
            fig_heat.update_layout(height=300, margin=dict(t=40, b=20))
            st.plotly_chart(fig_heat, use_container_width=True)

    else:
        st.info("Load `lng_arbitrage_clean.csv` with an `arbitrage_label` column to see the dashboard.")


2026-06-10 20:35:25.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [11]:

# Page Functions

def page_dashboard():
    st.title("📊 LNG Arbitrage Regime Dashboard")
    st.caption("Monthly regime classification based on JKM–TTF spread dynamics · Jan 2014 – Apr 2026")

    if df is None or "arbitrage_label" not in df.columns:
        st.info("Load `lng_arbitrage_clean.csv` with an `arbitrage_label` column to see the dashboard.")
        return

    # ── KPI Metric Cards ─────────────────────────────────────
    col1, col2, col3, col4 = st.columns(4)
    label_counts  = df["arbitrage_label"].value_counts()
    latest_spread = df["jkm_ttf_spread"].iloc[-1]
    latest_date   = df.index[-1].strftime("%b %Y")

    col1.metric("🔵 Asia Premium Months",   int(label_counts.get(2, 0)))
    col2.metric("🟠 Neutral Months",        int(label_counts.get(1, 0)))
    col3.metric("🔴 Europe Premium Months", int(label_counts.get(0, 0)))
    col4.metric(
        f"Latest Spread ({latest_date})",
        f"{latest_spread:.2f} $/MMBtu",
        delta=f"{(latest_spread - df['jkm_ttf_spread'].iloc[-2]):.2f} vs prior month",
    )

    st.divider()

    # ── Spread Over Time ──────────────────────────────────────
    fig_spread = go.Figure()
    for label_id, regime_name in LABEL_MAP.items():
        mask = df["arbitrage_label"] == label_id
        fig_spread.add_trace(go.Scatter(
            x=df.index[mask],
            y=df.loc[mask, "jkm_ttf_spread"],
            mode="markers+lines",
            name=regime_name,
            marker=dict(color=COLORS[regime_name], size=6),
            line=dict(color=COLORS[regime_name], width=1),
        ))

    fig_spread.add_hline(
        y=FREIGHT, line_dash="dash", line_color="#444",
        annotation_text=f"Asia threshold ({FREIGHT} $/MMBtu)",
        annotation_position="top left",
    )
    fig_spread.add_hline(
        y=-1.0, line_dash="dot", line_color="#888",
        annotation_text="Europe threshold (−1.0 $/MMBtu)",
        annotation_position="bottom left",
    )
    fig_spread.update_layout(
        title="JKM–TTF Spread Over Time — Coloured by Arbitrage Regime",
        xaxis_title="Date",
        yaxis_title="Spread ($/MMBtu)",
        legend_title="Regime",
        hovermode="x unified",
        height=420,
    )
    st.plotly_chart(fig_spread, use_container_width=True)

    # ── Regime Distribution + Annual Heatmap ─────────────────
    col_left, col_right = st.columns([1, 2])

    with col_left:
        fig_dist = go.Figure(go.Bar(
            x=[int(label_counts.get(2, 0)),
               int(label_counts.get(1, 0)),
               int(label_counts.get(0, 0))],
            y=["Asia Premium", "Neutral", "Europe Premium"],
            orientation="h",
            marker_color=[COLORS["Asia Premium"],
                          COLORS["Neutral"],
                          COLORS["Europe Premium"]],
        ))
        fig_dist.update_layout(
            title="Regime Distribution (all months)",
            xaxis_title="Count",
            height=300,
            margin=dict(t=40, b=20),
        )
        st.plotly_chart(fig_dist, use_container_width=True)

    with col_right:
        df_heat          = df[["arbitrage_label"]].copy()
        df_heat["year"]  = df_heat.index.year
        df_heat["month"] = df_heat.index.month
        pivot = df_heat.pivot(index="year", columns="month", values="arbitrage_label")
        month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                        "Jul","Aug","Sep","Oct","Nov","Dec"]

        fig_heat = px.imshow(
            pivot,
            color_continuous_scale=[
                [0.0, COLORS["Europe Premium"]],
                [0.5, COLORS["Neutral"]],
                [1.0, COLORS["Asia Premium"]],
            ],
            labels=dict(color="Regime"),
            x=month_labels[:pivot.shape[1]],
            y=pivot.index.tolist(),
            aspect="auto",
            title="Monthly Regime Heatmap (0=EP · 1=Neutral · 2=AP)",
        )
        fig_heat.update_layout(height=300, margin=dict(t=40, b=20))
        st.plotly_chart(fig_heat, use_container_width=True)


def page_live_predictor():
    st.title("🔍 Live Regime Predictor")
    st.caption("Adjust the sliders in the sidebar to simulate any market environment.")

    if model is None or scaler is None:
        st.error("Model not loaded. Ensure `models/best_model.pkl` and "
                 "`models/scaler.pkl` exist.")
        return

    label_int, label_str, proba_dict = predict(model, scaler, sidebar_inputs)

    # ── Regime Badge ──────────────────────────────────────────
    pill_class = {
        "Asia Premium":   "pill-asia",
        "Neutral":        "pill-neutral",
        "Europe Premium": "pill-europe",
    }[label_str]

    st.markdown(
        f'<h2>Predicted Regime: &nbsp;'
        f'<span class="{pill_class}">{label_str}</span></h2>',
        unsafe_allow_html=True,
    )

    # ── Probability Bar Chart ─────────────────────────────────
    fig_prob = go.Figure(go.Bar(
        x=list(proba_dict.keys()),
        y=list(proba_dict.values()),
        marker_color=[COLORS[k] for k in proba_dict],
        text=[f"{v:.1%}" for v in proba_dict.values()],
        textposition="outside",
    ))
    fig_prob.update_layout(
        title="Class Probabilities",
        yaxis=dict(range=[0, 1.15], tickformat=".0%"),
        height=320,
    )
    st.plotly_chart(fig_prob, use_container_width=True)

    # ── Spread Gauge ──────────────────────────────────────────
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number+delta",
        value=sidebar_inputs["jkm_ttf_spread"],
        delta={"reference": 0},
        gauge={
            "axis": {"range": [-10, 20]},
            "bar":  {"color": COLORS[label_str]},
            "steps": [
                {"range": [-10, -1.0],    "color": "#f5c0ba"},
                {"range": [-1.0, FREIGHT],"color": "#fde8cc"},
                {"range": [FREIGHT, 20],  "color": "#bdd6f0"},
            ],
            "threshold": {
                "line": {"color": "black", "width": 3},
                "thickness": 0.75,
                "value": sidebar_inputs["jkm_ttf_spread"],
            },
        },
        title={"text": "JKM–TTF Spread ($/MMBtu)"},
        number={"suffix": " $/MMBtu"},
    ))
    fig_gauge.update_layout(height=300)
    st.plotly_chart(fig_gauge, use_container_width=True)

    

# ── Threshold Explanation ─────────────────────
spread = sidebar_inputs["jkm_ttf_spread"]
st.divider()
st.subheader("How was this classified?")
st.markdown(f"""
| Condition | Threshold | Current Spread | Match |
|-----------|-----------|----------------|-------|
| JKM–TTF > {FREIGHT} | Asia Premium | {spread:.2f} | {"✅" if spread > FREIGHT else "❌"} |
| −1.0 ≤ JKM–TTF ≤ {FREIGHT} | Neutral | {spread:.2f} | {"✅" if -1.0 <= spread <= FREIGHT else "❌"} |
| JKM–TTF < −1.0 | Europe Premium | {spread:.2f} | {"✅" if spread < -1.0 else "❌"} |
    """)

In [12]:
def page_model_performance():
    st.title("📈 Model Performance Summary")
    st.caption("Test set: Jan 2023 – Apr 2026 (30 months)")

    results_json_path = "models/results.json"
    if os.path.exists(results_json_path):
        results_df = pd.read_json(results_json_path)
        st.success("✅ Loaded live results from `models/results.json`")
    else:
        results_df = pd.DataFrame({
            "Model":        ["KNN (K=3)", "SVM (RBF)", "Logistic Reg.", "Decision Tree",
                             "Random Forest", "XGBoost", "AdaBoost"],
            "Accuracy":     [0.800, 0.770, 0.670, 1.000, None, None, None],
            "Weighted F1":  [0.742, 0.670, 0.680, 1.000, None, None, None],
            "AP Recall":    [0.14,  0.00,  0.17,  1.00,  None, None, None],
            "EP Precision": [0.00,  0.00,  0.00,  1.00,  None, None, None],
        }).set_index("Model")
        st.info("💡 Showing placeholder results. Save `results_df.to_json('models/results.json')` from the notebook.")

    st.dataframe(
        results_df.style.format("{:.3f}", na_rep="—"),
        use_container_width=True,
    )

    st.divider()

    f1_vals = results_df["Weighted F1"].dropna()
    if not f1_vals.empty:
        fig_f1 = go.Figure(go.Bar(
            x=f1_vals.index.tolist(),
            y=f1_vals.values,
            marker_color=["#2980b9" if v == f1_vals.max() else "#bdc3c7"
                          for v in f1_vals.values],
            text=[f"{v:.3f}" for v in f1_vals.values],
            textposition="outside",
        ))
        fig_f1.add_hline(
            y=0.938, line_dash="dash", line_color="#e74c3c",
            annotation_text="DT baseline (0.938)",
            annotation_position="top right",
        )
        fig_f1.update_layout(
            title="Weighted F1 Score — All Models",
            yaxis=dict(range=[0, 1.15]),
            height=380,
        )
        st.plotly_chart(fig_f1, use_container_width=True)


def page_about():
    st.title("ℹ️ About This App")
    st.markdown

### Project Overview
This app classifies the **LNG arbitrage regime** for each month based on the JKM–TTF price spread.

| Regime | Spread Condition | Implication |
|--------|-----------------|-------------|
| 🔵 **Asia Premium** | > {FREIGHT} $/MMBtu | Send cargoes East |
| 🟠 **Neutral** | −1.0 to {FREIGHT} $/MMBtu | No directional edge |
| 🔴 **Europe Premium** | < −1.0 $/MMBtu | Send cargoes West |

---
### Dataset
- **Period:** January 2014 – April 2026 (147 months)
- **Train:** Jan 2014 – Dec 2022 · **Test:** Jan 2023 – Apr 2026
- **Features:** {len(FEATURES)} — spreads, price lags, rolling averages, LNG trade share


In [13]:
# ── Page Router ──────────────────────────────────────────────


PAGES = {
    "📊 Dashboard":        page_dashboard,
    "🔍 Live Predictor":   page_live_predictor,
    "📈 Model Performance": page_model_performance,
    "ℹ️ About":            page_about,
}

PAGES[page]()   # ← single call, no if/elif chain

2026-06-10 20:35:25.865 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.865 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.865 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.866 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.866 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.866 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.866 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 20:35:25.866 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar